# Chapter 9 — Crafting Elegant Code with Stacks and Queues

## Why this chapter is different from the previous ones

Chapters 3–8 were about **efficiency** (Big O, hash tables). This chapter is 
about something new: **code elegance and correctness through restriction**. 
Stacks and queues aren't faster than arrays — they're *arrays with rules*, 
and those rules are the entire point.

Both are examples of an **abstract data type**: a theoretical set of rules 
about how data should be accessed, implemented on top of some other real 
data structure (usually an array). The stack/queue doesn't care what's 
underneath — only that certain operations are allowed and others are forbidden.

## Stacks — LIFO (Last In, First Out)

Three restrictions, all operating on the same end (the "top"):
- Insert only at the top (**push**)
- Delete only from the top (**pop**)
- Read only the top element

Mental model: a stack of dishes — you can only add/remove/see the top one.

**LIFO** = the last item pushed is always the first item popped.

### Python stack build on list

In [ ]:
class Stack:
    def __init__(self):
        self.data = []

    def push(self, element):
        self.data.append(element)

    def pop(self):
        return self.data.pop()

    def read(self):
        return self.data[-1] if self.data else None


s = Stack()
s.push(5)
s.push(3)
s.push(0)
s.pop()      # 0 — last one in, first one out
s.pop()      # 3
s.read()     # 5 — only the top is visible

5

**Why wrap a list at all, if Python lists already support `.append()`/`.pop()`?** 
The whole value of the `Stack` class is that it **forbids** everything a 
plain list would otherwise allow — no `s.data[5]`, no removing from the 
middle. This restriction is a *feature*: it prevents a whole category of 
bugs, and it signals intent to anyone reading the code ("this is LIFO 
temporary data") without needing a comment to explain it.

## Stacks in action — the bracket-matching linter

The canonical stack use case: checking that opening/closing braces match 
correctly (`(`, `[`, `{`). This is directly relevant to the LeetCode 
problem, *Valid Parentheses*.

**The algorithm:**
1. Scan left to right, character by character.
2. See an **opening** brace → push it.
3. See a **closing** brace → pop the top of the stack and compare:
   - Stack was empty when you tried to pop → an unmatched closing brace (error).
   - Popped brace doesn't match the current closing brace type → mismatched 
     brace types (error).
   - Popped brace matches → good, continue.
4. End of string, but the stack still has items left → an unmatched opening 
   brace (error).
5. End of string, stack empty → valid.

**Why a stack specifically fits this problem**: the *most recently opened* 
brace must always be the *next one closed* — that's inherently a LIFO 
relationship, which is exactly what a stack enforces natively.

In [2]:
def is_valid_braces(text):
    stack = []
    pairs = {'(': ')', '[': ']', '{': '}'}

    for char in text:
        if char in pairs:                  # opening brace
            stack.append(char)
        elif char in pairs.values():         # closing brace
            if not stack:
                return f"{char} doesn't have opening brace"
            opening = stack.pop()
            if pairs[opening] != char:
                return f"{char} has mismatched opening brace"

    if stack:
        return f"{stack[-1]} does not have closing brace"

    return True


is_valid_braces("( var x = { y: [1, 2, 3] } )")   # True
is_valid_braces("var x = { y: [1, 2, 3] })")        # error message

") doesn't have opening brace"

## Queues — FIFO (First In, First Out)

Same idea as a stack — an array with restrictions — but a **different** set:
- Insert only at the **end** (**enqueue**) — same as a stack
- Delete only from the **front** (**dequeue**) — opposite of a stack
- Read only the **front** element — opposite of a stack

Mental model: a line at a movie theater — first person in line is the first 
one to enter.

**FIFO** = the first item added is always the first item removed.

In [3]:
class Queue:
    def __init__(self):
        self.data = []

    def enqueue(self, element):
        self.data.append(element)

    def dequeue(self):
        return self.data.pop(0)     # remove from the FRONT, not the end

    def read(self):
        return self.data[0] if self.data else None


q = Queue()
q.enqueue(5)
q.enqueue(9)
q.enqueue(100)
q.dequeue()     # 5 — first one in, first one out
q.dequeue()     # 9
q.read()          # 100

100

**Worth flagging (not explicitly in the book, but important)**: `list.pop(0)` is actually `O(N)` in Python,\
since every remaining element has to shift down one position — a plain list is a poor real-world choice for a queue at scale.\
Python's standard library provides `collections.deque`, purpose-built for `O(1)` operations at *both* ends — worth using that in real code instead of a hand-rolled list-based queue:

In [4]:
from collections import deque

q = deque()
q.append(5)          # enqueue -> O(1)
q.append(9)
q.popleft()            # dequeue -> O(1), unlike list.pop(0)

5

## Queues in action — processing jobs in order

Classic use case: a print queue. Jobs are added (`enqueue`) as they arrive, and printed (`dequeue`) strictly in the order received\
— guarantees fairness and correct sequencing for any "first come, first served" system (print jobs, background task workers, people waiting for a doctor, planes waiting for takeoff).

In [5]:
class PrintManager:
    def __init__(self):
        self.queue = Queue()

    def queue_print_job(self, document):
        self.queue.enqueue(document)

    def run(self):
        while self.queue.read() is not None:
            print(self.queue.dequeue())


pm = PrintManager()
pm.queue_print_job("First Document")
pm.queue_print_job("Second Document")
pm.queue_print_job("Third Document")
pm.run()
# First Document
# Second Document
# Third Document

First Document
Second Document
Third Document


## Why bother with a restricted structure, if an array can do everything a stack/queue can?

Three real reasons, straight from the book:

1. **Bug prevention** — if the linter algorithm accidentally removed from 
   the middle of an array instead of the top, the whole algorithm breaks 
   silently. A stack makes that mistake *impossible*, not just unlikely.
2. **A mental model for solving problems** — recognizing "this is a LIFO 
   problem" or "this is a FIFO problem" is itself a problem-solving tool, 
   independent of implementation.
3. **Communication with other developers** — seeing `Stack`/`Queue` (or a 
   `deque` used stack/queue-style) in code immediately tells a reader what 
   ordering guarantee is being relied on, without needing a comment.

## Exercises

1. **Conceptual**: a call center places callers on hold and assigns them to 
   "the next available representative" — should this use a stack or a queue?
2. **Trace by hand**: push `1, 2, 3, 4, 5, 6` onto a stack, then pop twice — 
   what value would you read next?
3. **Trace by hand**: insert `1, 2, 3, 4, 5, 6` into a queue, then dequeue 
   twice — what value would you read next?
4. **Implementation**: write a function that uses a stack to reverse a 
   string (e.g. `"abcde"` → `"edcba"`) — use the `Stack` class from this chapter.